# Learning Spatial Relationships with MISTy on gut spatial data (stem cell niche 2)
**Developed by:** Anna Maguza  
**Affiliation:** Faculty of Medicine, Würzburg University  
**Creation date:** 9th April 2025  
**Last modified date:** 9th April 2025  

In this notebook, we:  
  
* Calculate pathway activities with decoupler  
* Run MISTy to understand how cell surrounding influence on cell behaviour  
* Analyze pathway, transcription factors and CCI influence  

## Import packages

In [1]:
import scanpy as sc
import decoupler as dc
import plotnine as p9
import liana as li
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad

import json
from datetime import datetime
import squidpy as sq

/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/sin

In [2]:
from liana.method import MistyData, genericMistyData, lrMistyData
from liana.method.sp import RandomForestModel, LinearModel, RobustLinearModel

## Set up notebooks

In [3]:
timestamp = datetime.now().strftime('%d%m%Y_%H%M%S')

In [4]:
plt.rcParams['figure.dpi'] = 300
plt.rcParams['figure.figsize'] = (15, 15)

## Load imputed Xenium data

In [5]:
adata = sc.read_h5ad('data/gut_data/maxfuse_checkpoints/gut_hs_XeniumHealthyColon_maxfuse_imputed_and_originalcounts_updated_niches_09042025_131844_log.h5ad')

+ Leave only niche 2

In [6]:
adata = adata[adata.obs['latent_leiden_0.4'] == '2'].copy()

In [7]:
adata

AnnData object with n_obs × n_vars = 25249 × 10281
    obs: 'Study_name', 'Donor_ID', 'Library_Preparation_Protocol', 'dataset', '_scvi_batch', '_scvi_labels', 'seed_labels', 'C_scANVI', 'SC_subsets', 'Cell_State', 'cell_id', 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'n_counts', 'REG4_score', 'gdT', 'Endothelial cells', 'latent_leiden_0.4', 'CD24_ligand_receptor_target_gene_GP', 'SLPI_ligand_receptor_target_gene_GP', 'CXCL14_ligand_receptor_target_gene_GP', 'ANPEP_ligand_receptor_target_gene_GP', 'IL1B_ligand_receptor_target_gene_GP', 'TIMP3_ligand_receptor_target_gene_GP', 'CDH1_ligand_receptor_target_gene_GP', 'TNXB_ligand_receptor_target_gene_GP', 'CLU_ligand_receptor_target_gene_GP', 'TFF1_ligand_receptor_target_gene_GP', 'CCL11_ligand_receptor_target_gene_GP', 'ROBO1_ligand_receptor_target_gene_GP', 'NRG3_ligand_receptor

+ Delete cells where imputation didnt actually work

In [8]:
adata = adata[adata.obs['has_match'] == True].copy()

In [9]:
adata

AnnData object with n_obs × n_vars = 15621 × 10281
    obs: 'Study_name', 'Donor_ID', 'Library_Preparation_Protocol', 'dataset', '_scvi_batch', '_scvi_labels', 'seed_labels', 'C_scANVI', 'SC_subsets', 'Cell_State', 'cell_id', 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'n_counts', 'REG4_score', 'gdT', 'Endothelial cells', 'latent_leiden_0.4', 'CD24_ligand_receptor_target_gene_GP', 'SLPI_ligand_receptor_target_gene_GP', 'CXCL14_ligand_receptor_target_gene_GP', 'ANPEP_ligand_receptor_target_gene_GP', 'IL1B_ligand_receptor_target_gene_GP', 'TIMP3_ligand_receptor_target_gene_GP', 'CDH1_ligand_receptor_target_gene_GP', 'TNXB_ligand_receptor_target_gene_GP', 'CLU_ligand_receptor_target_gene_GP', 'TFF1_ligand_receptor_target_gene_GP', 'CCL11_ligand_receptor_target_gene_GP', 'ROBO1_ligand_receptor_target_gene_GP', 'NRG3_ligand_receptor

+ Visualize the dataset with filtered niche

In [ ]:
plt.rcParams['figure.dpi'] = 300
plt.rcParams['figure.figsize'] = (7, 7)

# Your plotting code
sq.pl.spatial_scatter(
    adata,
    library_id="spatial",
    shape=None, 
    color=['C_scANVI', 'match_score'],
    size=1,
    frameon=False,
    alpha=1.0,
    cmap = 'magma_r',
    ncols=1,
    #crop_coord=[(6300, 1950, 7600, 2950)]

)
plt.savefig(f"figures/niche2_xenium_after_imputation.png", bbox_inches="tight", dpi=300)

## Funconomics

In [11]:
progeny = dc.get_progeny(organism='human', top=500)

In [12]:
dc.run_mlm(
    mat=adata,
    net=progeny,
    source='source',
    target='target',
    weight='weight',
    verbose=True,
    use_raw=False,
)

183 features of mat are empty, they will be removed.
Running mlm on mat with 15621 samples and 10098 targets for 14 sources.


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:02<00:00,  1.45s/it]


In [13]:
acts_progeny = li.ut.obsm_to_adata(adata, 'mlm_estimate')

In [14]:
acts_progeny.var

""
Androgen
EGFR
Estrogen
Hypoxia
JAK-STAT
MAPK
NFkB
PI3K
TGFb
TNFa


In [ ]:
plt.rcParams['figure.dpi'] = 300
plt.rcParams['figure.figsize'] = (7, 7)

# Your plotting code
sq.pl.spatial_scatter(
    acts_progeny,
    library_id="spatial",
    shape=None, 
    color=['JAK-STAT', 'Androgen','Hypoxia', 'MAPK', 'EGFR', 'Estrogen'],
    size=1,
    frameon=False,
    alpha=1.0,
    cmap = 'RdBu_r',
    ncols=3,
    #crop_coord=[(6300, 1950, 7600, 2950)]

)
plt.savefig(f"figures/niche2_pathways_activity_in_spatial1.png", bbox_inches="tight", dpi=300)

In [ ]:
plt.rcParams['figure.dpi'] = 300
plt.rcParams['figure.figsize'] = (7, 7)

# Your plotting code
sq.pl.spatial_scatter(
    acts_progeny,
    library_id="spatial",
    shape=None, 
    color=['TGFb', 'TNFa', 'Trail', 'NFkB', 'PI3K', 'WNT', 'VEGF', 'p53'],
    size=1,
    frameon=False,
    alpha=1.0,
    cmap = 'RdBu_r',
    ncols=4,
    #crop_coord=[(6300, 1950, 7600, 2950)]

)
plt.savefig(f"figures/niche2_pathways_activity_in_spatial2.png", bbox_inches="tight", dpi=300)

## Formatting & Running MISTy

In [15]:
cell_types = adata.obs['C_scANVI'].unique()
print(f"Found {len(cell_types)} cell types: {cell_types}")

Found 24 cell types: ['Mast cells', 'Endothelial cells', 'Goblet cells', 'Tuft cells', 'EECs', ..., 'DC', 'Colonocyte', 'NK', 'Monocytes', 'BEST4+ epithelial']
Length: 24
Categories (24, object): ['B cells', 'BEST4+ epithelial', 'CD4 T', 'CD8 T', ..., 'Plasma cells', 'Stem cells', 'TA', 'Tuft cells']


In [16]:
compositions = pd.DataFrame(index=adata.obs_names, columns=cell_types)

In [17]:
compositions = compositions.fillna(0)

/var/folders/kr/nd4y_1_s34n42lrht8wdh4cr0000gq/T/ipykernel_40151/2262236102.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


In [18]:
for cell_type in cell_types:
    mask = adata.obs['C_scANVI'] == cell_type
    compositions.loc[mask, cell_type] = 1

In [19]:
adata.obsm['compositions'] = compositions

In [20]:
comps = ad.AnnData(
    X=adata.obsm['compositions'].values,
    obs=adata.obs,
    var=pd.DataFrame(index=adata.obsm['compositions'].columns),
    obsm=adata.obsm,
)

In [21]:
misty = genericMistyData(intra=comps, extra=acts_progeny, cutoff=0.05, bandwidth=200, n_neighs=6)

/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/anndata/_core/anndata.py:401: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/anndata/_core/anndata.py:401: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/anndata/_core/anndata.py:401: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns fro

## Learn Relationships with MISTy

In [22]:
# Run this if you have time and resources
misty(model=RandomForestModel, n_jobs=-1, verbose = True, seed=1337)

Now learning: Stem cells: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:23<00:00,  7.75s/it]


In [ ]:
# Run this if you want it much faster and bit more simplistic 
#misty(model=LinearModel, k_cv=10, seed=1337, bypass_intra=True, verbose = True)

In [23]:
misty.uns['target_metrics'].head()

,target,intra_R2,multi_R2,gain_R2,intra,juxta,para
0,Goblet cells,0.145044,0.187104,0.042060,0.574741,0.283227,0.142031
1,Tuft cells,0.127314,0.165065,0.037751,0.571260,0.220752,0.207988
2,Stem cells,0.196757,0.308863,0.112106,0.479993,0.335247,0.184760


In [24]:
fig = (li.pl.target_metrics(misty, stat='intra_R2', return_fig=True))
print(fig)
fig.save("figures/niche2_misty_intra_R2.png", dpi=300)

/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/plotnine/ggplot.py:615: PlotnineWarning: Saving 5 x 5 in image.
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/plotnine/ggplot.py:616: PlotnineWarning: Filename: /Users/annamaguza/Desktop/Repos/Fetal_stem_cells_analysis/6_cci_on_imputed_spatial/figures/niche2_misty_intra_R2.png


<ggplot: (500 x 500)>


In [25]:
fig = (li.pl.target_metrics(misty, stat='gain_R2'))
print(fig)
fig.save("figures/niche2_misty_gain_R2.png", dpi=300)

<ggplot: (500 x 500)>


/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/plotnine/ggplot.py:615: PlotnineWarning: Saving 5 x 5 in image.
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/plotnine/ggplot.py:616: PlotnineWarning: Filename: /Users/annamaguza/Desktop/Repos/Fetal_stem_cells_analysis/6_cci_on_imputed_spatial/figures/niche2_misty_gain_R2.png


In [26]:
fig = (li.pl.contributions(misty, return_fig=True))
print(fig)
fig.save("figures/niche2_misty_contributions.png", dpi=300)

<ggplot: (500 x 500)>


/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/plotnine/ggplot.py:615: PlotnineWarning: Saving 5 x 5 in image.
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/plotnine/ggplot.py:616: PlotnineWarning: Filename: /Users/annamaguza/Desktop/Repos/Fetal_stem_cells_analysis/6_cci_on_imputed_spatial/figures/niche2_misty_contributions.png


In [27]:
misty.uns['interactions'].head()

,target,predictor,view,importances
0,Goblet cells,Tuft cells,intra,0.349089
1,Goblet cells,Stem cells,intra,0.650911
16,Tuft cells,Goblet cells,intra,0.407265
17,Tuft cells,Stem cells,intra,0.592735
32,Stem cells,Goblet cells,intra,0.483182


In [28]:
fig = ((
    li.pl.interactions(misty, view='juxta', return_fig=True, figure_size=(7,5)) +
    p9.scale_fill_gradient2(low = "blue", mid = "white", high = "red", midpoint = 0)
))
print(fig)
fig.save("figures/niche2_misty_pathway_importance.png", dpi=300)

<ggplot: (700 x 500)>


/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/plotnine/ggplot.py:615: PlotnineWarning: Saving 7 x 5 in image.
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/plotnine/ggplot.py:616: PlotnineWarning: Filename: /Users/annamaguza/Desktop/Repos/Fetal_stem_cells_analysis/6_cci_on_imputed_spatial/figures/niche2_misty_pathway_importance.png


## Estimate Transcription Factor activities with decoupler

In [29]:
net = dc.get_collectri()

In [30]:
dc.run_ulm(
    mat=adata,
    net=net,
    verbose=True,
    use_raw=False,
)

183 features of mat are empty, they will be removed.
Running ulm on mat with 15621 samples and 10098 targets for 590 sources.


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:01<00:00,  1.16it/s]


In [31]:
acts_tfs = li.ut.obsm_to_adata(adata, 'ulm_estimate')

In [32]:
acts_tfs.var

""
AEBP1
AHR
AIP
AIRE
AP1
...
ZNF382
ZNF384
ZNF699
ZNF804A


In [33]:
li.ut.spatial_neighbors(acts_tfs, cutoff=0.1, bandwidth=200, set_diag=True)

In [34]:
fig = (li.pl.connectivity(acts_tfs, idx=0, figure_size=(10,10)))
print(fig)
fig.save("figures/niche2_misty_liana_connectivity.png", dpi=300)

<ggplot: (1000 x 1000)>


/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/plotnine/ggplot.py:615: PlotnineWarning: Saving 10 x 10 in image.
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/plotnine/ggplot.py:616: PlotnineWarning: Filename: /Users/annamaguza/Desktop/Repos/Fetal_stem_cells_analysis/6_cci_on_imputed_spatial/figures/niche2_misty_liana_connectivity.png


In [35]:
acts_progeny.obsm['spatial'] = acts_tfs.obsm['spatial']
acts_progeny.obsp['spatial_connectivities'] = acts_tfs.obsp['spatial_connectivities']

In [36]:
misty = MistyData(data={"intra": comps, "TFs": acts_tfs, "Pathways": acts_progeny})

/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/mudata/_core/mudata.py:1429: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.


In [37]:
misty(model=LinearModel, verbose=True, bypass_intra=True)

Now learning: BEST4+ epithelial: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 24/24 [01:02<00:00,  2.62s/it]


In [38]:
fig = (li.pl.target_metrics(misty, stat='gain_R2'))
print(fig)
fig.save("figures/niche2_misty_TF-and_pathway_gain_R2.png", dpi=300)

/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/plotnine/ggplot.py:615: PlotnineWarning: Saving 5 x 5 in image.
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/plotnine/ggplot.py:616: PlotnineWarning: Filename: /Users/annamaguza/Desktop/Repos/Fetal_stem_cells_analysis/6_cci_on_imputed_spatial/figures/niche2_misty_TF-and_pathway_gain_R2.png


<ggplot: (500 x 500)>


In [39]:
fig = (li.pl.contributions(misty, return_fig=True))
print(fig)
fig.save("figures/niche2_misty_contributions_TFs_and_pathways.png", dpi=300)

<ggplot: (500 x 500)>


/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/plotnine/ggplot.py:615: PlotnineWarning: Saving 5 x 5 in image.
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/plotnine/ggplot.py:616: PlotnineWarning: Filename: /Users/annamaguza/Desktop/Repos/Fetal_stem_cells_analysis/6_cci_on_imputed_spatial/figures/niche2_misty_contributions_TFs_and_pathways.png
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/plotnine/layer.py:344: PlotnineWarning: position_stack : Removed 2 rows containing missing values.


In [40]:
fig = (
li.pl.interactions(misty, view='TFs', top_n=20) +
    p9.labs(x='Transcription Factor', y='Cell type') +
    p9.theme_bw(base_size=14) +
    p9.theme(axis_text_x=p9.element_text(rotation=90, size=13)) +
    # change to blue-red
    p9.scale_fill_gradient2(low='blue', mid='white', high='red')
)

# Display the figure
print(fig)

# Save the figure to a file
fig.save("figures/niche2_misty_TFs.png", dpi=300)

<ggplot: (640 x 480)>


/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/plotnine/ggplot.py:615: PlotnineWarning: Saving 6.4 x 4.8 in image.
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/plotnine/ggplot.py:616: PlotnineWarning: Filename: /Users/annamaguza/Desktop/Repos/Fetal_stem_cells_analysis/6_cci_on_imputed_spatial/figures/niche2_misty_TFs.png


## Ligand-Receptor Misty

In [41]:
misty = lrMistyData(adata, bandwidth=200, set_diag=False, cutoff=0.01, nz_threshold=0.1)

/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/anndata/_core/anndata.py:401: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/anndata/_core/anndata.py:401: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/anndata/_core/anndata.py:401: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns fro

In [42]:
misty(bypass_intra=True, model=LinearModel, verbose=True)

Now learning: IFNAR2: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 415/415 [14:48<00:00,  2.14s/it]


In [43]:
fig = (
    li.pl.interactions(misty, view='extra', return_fig=True, figure_size=(6, 5), top_n=25, key=abs) +
    p9.scale_fill_gradient2(low="blue", mid="white", high="red", midpoint=0) +
    p9.labs(y='Receptor', x='Ligand')
)

# Display the figure
print(fig)

# Save the figure to a file
fig.save("figures/niche2_misty_ligand_receptors_interactions.png", dpi=300)

<ggplot: (600 x 500)>


/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/plotnine/ggplot.py:615: PlotnineWarning: Saving 6 x 5 in image.
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/plotnine/ggplot.py:616: PlotnineWarning: Filename: /Users/annamaguza/Desktop/Repos/Fetal_stem_cells_analysis/6_cci_on_imputed_spatial/figures/niche2_misty_ligand_receptors_interactions.png
